In [ ]:
import os
import pandas as pd
import numpy as np


def load_and_combine_csvs(files, period_name=None):
    """
    Load multiple CSV files and combine them into one DataFrame.

    Parameters
    ----------
    files : list of str
        CSV file paths.
    period_name : str, optional
        Label for the period, added as a column.

    Returns
    -------
    pd.DataFrame
    """
    dfs = []

    for file in files:
        df = pd.read_csv(file, low_memory=False, encoding="latin1")

        filename = os.path.basename(file)
        year = filename[:4]

        df["year"] = pd.to_numeric(year, errors="coerce")

        if period_name is not None:
            df["period"] = period_name

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


def clean_flight_data(df):
    """
    Clean flight dataset and convert relevant columns to numeric.
    """
    df = df.copy()
    df = df.replace("NA", np.nan)

    numeric_cols = [
        "AirTime",
        "ActualElapsedTime",
        "CRSElapsedTime",
        "ArrDelay",
        "DepDelay",
        "Distance",
        "Cancelled",
        "Diverted",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    if "Distance" in df.columns:
        df["Distance_km"] = df["Distance"] * 1.60934

    df = df.drop(columns=["Distance"])
    df = df.rename(columns={"Distance_km": "Distance"})
        
    return df


def build_airport_nodes(df):
    """
    Create airport-level node analysis table.

    Each row is one airport (IATA code), with counts and average metrics.
    """
    required_cols = ["Origin", "Dest"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    origin_stats = (
        df.groupby("Origin")
        .agg(
            origin_flight_count=("Origin", "size"),
            avg_actual_elapsed_time=("ActualElapsedTime", "mean"),
            avg_scheduled_elapsed_time=("CRSElapsedTime", "mean"),
            avg_air_time=("AirTime", "mean"),
            avg_distance=("Distance", "mean"),
            avg_dep_delay=("DepDelay", "mean"),
            cancelled_origin_count=("Cancelled", "sum"),
            diverted_origin_count=("Diverted", "sum"),
            unique_destinations=("Dest", "nunique"),
        )
        .reset_index()
        .rename(columns={"Origin": "IATA"})
    )

    dest_stats = (
        df.groupby("Dest")
        .agg(
            dest_flight_count=("Dest", "size"),
            avg_arr_delay=("ArrDelay", "mean"),
            unique_origins=("Origin", "nunique"),
        )
        .reset_index()
        .rename(columns={"Dest": "IATA"})
    )

    airport_nodes = pd.merge(origin_stats, dest_stats, on="IATA", how="outer")

    count_cols = [
        "origin_flight_count",
        "dest_flight_count",
        "cancelled_origin_count",
        "diverted_origin_count",
        "unique_destinations",
        "unique_origins",
    ]

    for col in count_cols:
        airport_nodes[col] = airport_nodes[col].fillna(0)

    airport_nodes["flight_count"] = (
        airport_nodes["origin_flight_count"] + airport_nodes["dest_flight_count"]
    )

    airport_nodes["unique_connections"] = (
        airport_nodes["unique_destinations"] + airport_nodes["unique_origins"]
    )

    airport_nodes["avg_flight_time"] = airport_nodes["avg_actual_elapsed_time"]

    round_cols = [
        "avg_flight_time",
        "avg_actual_elapsed_time",
        "avg_scheduled_elapsed_time",
        "avg_air_time",
        "avg_distance",
        "avg_dep_delay",
        "avg_arr_delay",
    ]

    for col in round_cols:
        airport_nodes[col] = airport_nodes[col].round(2)

    airport_nodes = airport_nodes[
        [
            "IATA",
            "flight_count",
            "origin_flight_count",
            "dest_flight_count",
            "avg_flight_time",
            "avg_actual_elapsed_time",
            "avg_scheduled_elapsed_time",
            "avg_air_time",
            "avg_distance",
            "avg_dep_delay",
            "avg_arr_delay",
            "cancelled_origin_count",
            "diverted_origin_count",
            "unique_destinations",
            "unique_origins",
            "unique_connections",
        ]
    ].sort_values("flight_count", ascending=False)

    return airport_nodes


def save_airport_nodes_for_period(files, period_name, output_dir="."):
    """
    Parameters
    ----------
    files : list of str
        CSV files to combine for the period.
    period_name : str
        Name of the period, used in the output filename.
    output_dir : str
        Directory to save the CSV.

    Returns
    -------
    pd.DataFrame
    """
    df = load_and_combine_csvs(files, period_name=period_name)
    df = clean_flight_data(df)
    airport_nodes = build_airport_nodes(df)

    safe_period_name = period_name.replace(" ", "_").replace("-", "_")
    output_path = os.path.join(output_dir, f"airport_nodes_{safe_period_name}.csv")

    airport_nodes.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

    return airport_nodes


period_1999_2000_files = ["/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/1999.csv", "/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2000.csv"]
period_2002_2003_files = ["/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2002.csv", "/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2003.csv"]

nodes_1999_2000 = save_airport_nodes_for_period(
    period_1999_2000_files,
    "1999-2000"
)

nodes_2002_2003 = save_airport_nodes_for_period(
    period_2002_2003_files,
    "2002-2003"
)


Saved: ./airport_nodes_1999_2000.csv
Saved: ./airport_nodes_2002_2003.csv


In [113]:
nodes_2002_2003

,IATA,flight_count,origin_flight_count,dest_flight_count,avg_flight_time,avg_actual_elapsed_time,avg_scheduled_elapsed_time,avg_air_time,avg_distance,avg_dep_delay,avg_arr_delay,cancelled_origin_count,diverted_origin_count,unique_destinations,unique_origins,unique_connections
202,ORD,1388214.0,694093.0,694121.0,129.96,129.96,130.98,105.26,1216.27,7.84,5.53,15285.0,1104.0,132.0,130.0,262.0
77,DFW,1228058.0,613878.0,614180.0,129.08,129.08,131.45,108.28,1235.17,5.61,1.01,8413.0,884.0,139.0,135.0,274.0
16,ATL,1201228.0,600671.0,600557.0,116.82,116.82,116.45,84.20,1029.34,7.90,5.82,6794.0,950.0,150.0,150.0,300.0
149,LAX,812923.0,406372.0,406551.0,147.16,147.16,150.62,131.84,1617.66,4.29,0.75,4480.0,510.0,78.0,76.0,154.0
212,PHX,687674.0,343837.0,343837.0,136.54,136.54,140.27,118.53,1453.29,6.93,1.57,3444.0,569.0,72.0,68.0,140.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,FMN,4.0,2.0,2.0,33.00,33.00,69.00,23.50,238.18,185.50,NaN,0.0,0.0,1.0,2.0,3.0
221,PUB,2.0,0.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,1.0,1.0
24,BFF,2.0,1.0,1.0,41.00,41.00,79.00,194.00,244.62,131.00,NaN,0.0,0.0,1.0,1.0,2.0
70,CYS,2.0,1.0,1.0,40.00,40.00,25.00,22.00,144.84,185.00,NaN,0.0,0.0,1.0,1.0,2.0


In [92]:
def build_edge_list(df):
    """
    Create weighted edges between airports.
    Each row = connection between two airports
    """

    edges = (
        df.groupby(["Origin", "Dest"])
        .agg(
            flight_count=("Origin", "size"),
            avg_distance=("Distance", "mean"),
            avg_air_time=("AirTime", "mean"),
            avg_dep_delay=("DepDelay", "mean"),
            avg_arr_delay=("ArrDelay", "mean"),
        )
        .reset_index()
    )
    round_cols = ["avg_distance", "avg_air_time", "avg_dep_delay", "avg_arr_delay"]

    edges[round_cols] = edges[round_cols].round(2)

    return edges

In [93]:
df_1999_2000 = load_and_combine_csvs(["/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/1999.csv", "/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2000.csv"], "1999-2000")
#df_1999_2000 = clean_flight_data(df_1999_2000)

edges_1999_2000 = build_edge_list(df_1999_2000)
#edges_1999_2000.to_csv("edges_1999_2000.csv", index=False)

df_2002_2003 = load_and_combine_csvs(["/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2002.csv", "/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2003.csv"], "2002_2003")
#df_2002_2003 = clean_flight_data(df_2002_2003)

edges_2002_2003 = build_edge_list(df_2002_2003)
#edges_2002_2003.to_csv("edges_2002_2003.csv", index=False)

In [94]:
print(len(edges_2002_2003))
print(len(edges_1999_2000))

4607
3472


In [95]:
print(sum(nodes_1999_2000["cancelled_origin_count"]))
print(sum(nodes_2002_2003["cancelled_origin_count"]))

print(sum(nodes_1999_2000["diverted_origin_count"]))
print(sum(nodes_2002_2003["diverted_origin_count"]))

341801
166612.0
27809
19737.0


In [96]:
import pandas as pd
import networkx as nx


def build_airport_graph(nodes_df, edges_df, directed=True):
    if directed:
        G = nx.DiGraph()
    else:
        G = nx.Graph()

    # Add nodes with attributes
    for _, row in nodes_df.iterrows():
        node_id = row["IATA"]
        attrs = row.drop("IATA").to_dict()
        G.add_node(node_id, **attrs)

    # Add edges with attributes
    for _, row in edges_df.iterrows():
        source = row["Origin"]
        target = row["Dest"]
        attrs = row.drop(["Origin", "Dest"]).to_dict()
        G.add_edge(source, target, **attrs)

    return G

In [97]:
nodes_df = pd.read_csv("airport_nodes_1999_2000.csv")
edges_df = pd.read_csv("edges_1999_2000.csv")

G_before = build_airport_graph(nodes_df, edges_df, directed=True)

print("Number of nodes:", G_before.number_of_nodes())
print("Number of edges:", G_before.number_of_edges())

Number of nodes: 208
Number of edges: 3472


In [98]:
nodes_df = pd.read_csv("airport_nodes_2002_2003.csv")
edges_df = pd.read_csv("edges_2002_2003.csv")

G_after = build_airport_graph(nodes_df, edges_df, directed=True)

print("Number of nodes:", G_after.number_of_nodes())
print("Number of edges:", G_after.number_of_edges())

Number of nodes: 285
Number of edges: 4607


In [99]:
print(nx.numeric_assortativity_coefficient(G_after, "flight_count"))
print(nx.numeric_assortativity_coefficient(G_before, "flight_count"))


-0.2696023570094179
-0.2764237147499827


In [100]:
G_before.number_of_edges()

3472

In [101]:
density = 2 * G_before.number_of_edges() / (G_before.number_of_nodes() * (G_before.number_of_nodes() - 1))
print(density)

print(nx.density(G_before))

0.16127833519137866
0.08063916759568933


## network visualization

In [102]:
import networkx as nx
from netwulf import visualize
import ast
import pandas as pd
import itertools as iter
from collections import Counter
import matplotlib
matplotlib.use("Agg")


In [103]:
strength_dict = dict(G_before.degree(weight="weight"))
nx.set_node_attributes(G_before, strength_dict, "strength")

In [116]:
import pandas as pd
import numpy as np
import networkx as nx
from netwulf import visualize

# 1. Load edge data
df_edges = pd.read_csv("edges_2002_2003.csv")

# Build weighted graph
G = nx.Graph()

for _, row in df_edges.iterrows():
    origin = row["Origin"]
    dest   = row["Dest"]
    weight = row["flight_count"]
    if G.has_edge(origin, dest):
        G[origin][dest]["weight"] += weight
    else:
        G.add_edge(origin, dest,
                   weight          = weight,
                   avg_distance    = row.get("avg_distance"),
                   avg_air_time    = row.get("avg_air_time"),
                   avg_dep_delay   = row.get("avg_dep_delay"),
                   avg_arr_delay   = row.get("avg_arr_delay"))

# 2. Load node metadata
df_nodes = pd.read_csv("/Users/haseebshafi/Documents/GitHub/Computational-Social-Science-2026/Final_project/Graph/airport_nodes_2002_2003.csv")

key_col = "IATA"
df_nodes = df_nodes[df_nodes[key_col].isin(G.nodes())].copy()
df_nodes = df_nodes.replace([np.inf, -np.inf], np.nan)
df_nodes = df_nodes.astype(object).where(pd.notnull(df_nodes), None)

attr_dict = df_nodes.set_index(key_col).to_dict(orient="index")
nx.set_node_attributes(G, attr_dict)

#airport region dictionary, creatted with help of AI. 
airport_region = {
    # Northeast
    "BOS": "Northeast", "JFK": "Northeast", "LGA": "Northeast",
    "EWR": "Northeast", "PHL": "Northeast", "BWI": "Northeast",
    "DCA": "Northeast", "IAD": "Northeast", "BDL": "Northeast",
    "PVD": "Northeast", "MHT": "Northeast", "BUF": "Northeast",
    "ALB": "Northeast", "SYR": "Northeast", "ROC": "Northeast",
    "MDT": "Northeast", "ISP": "Northeast", "ABE": "Northeast",
    "PWM": "Northeast", "HPN": "Northeast", "BTV": "Northeast",
    "AVP": "Northeast", "ITH": "Northeast", "ELM": "Northeast",
    "SWF": "Northeast", "ERI": "Northeast", "BGM": "Northeast",
    "BGR": "Northeast", "ORH": "Northeast", "ACK": "Northeast",
    "ACY": "Northeast", "SCE": "Northeast",

    # Southeast
    "ATL": "Southeast", "MCO": "Southeast", "MIA": "Southeast",
    "FLL": "Southeast", "TPA": "Southeast", "CLT": "Southeast",
    "RDU": "Southeast", "ORF": "Southeast", "BNA": "Southeast",
    "MEM": "Southeast", "MSY": "Southeast", "JAX": "Southeast",
    "PBI": "Southeast", "RSW": "Southeast", "SRQ": "Southeast",
    "BHM": "Southeast", "RIC": "Southeast", "GSO": "Southeast",
    "LIT": "Southeast", "JAN": "Southeast", "CHS": "Southeast",
    "GSP": "Southeast", "TYS": "Southeast", "PNS": "Southeast",
    "HSV": "Southeast", "SAV": "Southeast", "CAE": "Southeast",
    "BTR": "Southeast", "MOB": "Southeast", "SHV": "Southeast",
    "LEX": "Southeast", "MYR": "Southeast", "ROA": "Southeast",
    "TLH": "Southeast", "DAB": "Southeast", "MLU": "Southeast",
    "ILM": "Southeast", "MLB": "Southeast", "AVL": "Southeast",
    "FAY": "Southeast", "TRI": "Southeast", "CRW": "Southeast",
    "AGS": "Southeast", "VPS": "Southeast", "GPT": "Southeast",
    "CHA": "Southeast", "MGM": "Southeast", "SDF": "Southeast",
    "LWB": "Southeast", "LFT": "Southeast", "PIE": "Southeast",
    "EYW": "Southeast", "GNV": "Southeast", "CSG": "Southeast",
    "GTR": "Southeast", "LYH": "Southeast", "HTS": "Southeast",
    "FLO": "Southeast", "DHN": "Southeast", "LCH": "Southeast",
    "VLD": "Southeast", "MCN": "Southeast", "BQK": "Southeast",
    "ABY": "Southeast", "MEI": "Southeast",

    # Midwest
    "ORD": "Midwest",  "MDW": "Midwest",  "DTW": "Midwest",
    "MSP": "Midwest",  "STL": "Midwest",  "MKE": "Midwest",
    "CLE": "Midwest",  "CMH": "Midwest",  "IND": "Midwest",
    "CVG": "Midwest",  "PIT": "Midwest",  "DSM": "Midwest",
    "OMA": "Midwest",  "MCI": "Midwest",  "MSN": "Midwest",
    "DAY": "Midwest",  "GRR": "Midwest",  "ICT": "Midwest",
    "CID": "Midwest",  "MBS": "Midwest",  "FSD": "Midwest",
    "LNK": "Midwest",  "GRB": "Midwest",  "SBN": "Midwest",
    "FAR": "Midwest",  "LAN": "Midwest",  "RST": "Midwest",
    "SGF": "Midwest",  "MLI": "Midwest",  "AZO": "Midwest",
    "FNT": "Midwest",  "RAP": "Midwest",  "DLH": "Midwest",
    "GFK": "Midwest",  "MOT": "Midwest",  "TVC": "Midwest",
    "CAK": "Midwest",  "TOL": "Midwest",  "SUX": "Midwest",
    "LSE": "Midwest",  "FWA": "Midwest",  "PIA": "Midwest",
    "MIB": "Midwest",  "BIS": "Midwest",  "DBQ": "Midwest",
    "ATW": "Midwest",  "MQT": "Midwest",  "BFF": "Midwest",

    # Southwest
    "DFW": "Southwest", "DAL": "Southwest", "HOU": "Southwest",
    "IAH": "Southwest", "PHX": "Southwest", "TUS": "Southwest",
    "ABQ": "Southwest", "SAT": "Southwest", "AUS": "Southwest",
    "ELP": "Southwest", "OKC": "Southwest", "TUL": "Southwest",
    "MAF": "Southwest", "LBB": "Southwest", "HRL": "Southwest",
    "MFE": "Southwest", "CRP": "Southwest", "AMA": "Southwest",
    "BRO": "Southwest", "BPT": "Southwest", "VCT": "Southwest",
    "EFD": "Southwest", "FMN": "Southwest", "YUM": "Southwest",

    # West
    "LAX": "West", "SFO": "West", "SJC": "West", "OAK": "West",
    "SEA": "West", "PDX": "West", "LAS": "West", "SAN": "West",
    "SMF": "West", "SNA": "West", "BUR": "West", "LGB": "West",
    "ONT": "West", "RNO": "West", "BZN": "West", "SLC": "West",
    "PSP": "West", "EUG": "West", "SBA": "West", "PSC": "West",
    "MFR": "West", "MRY": "West", "FAT": "West", "BFL": "West",
    "RDD": "West", "CLD": "West", "SMX": "West", "CEC": "West",
    "CIC": "West", "IYK": "West", "VIS": "West", "IPL": "West",
    "MOD": "West", "OXR": "West",

    # Mountain
    "DEN": "Mountain", "BOI": "Mountain", "GEG": "Mountain",
    "BIL": "Mountain", "FCA": "Mountain", "JAC": "Mountain",
    "COS": "Mountain", "MSO": "Mountain", "GTF": "Mountain",
    "EGE": "Mountain", "HLN": "Mountain", "HDN": "Mountain",
    "MTJ": "Mountain", "GUC": "Mountain", "DRO": "Mountain",
    "CPR": "Mountain", "CDC": "Mountain", "COD": "Mountain",
    "WYS": "Mountain", "TWF": "Mountain", "PIH": "Mountain",
    "BTM": "Mountain", "PUB": "Mountain", "CYS": "Mountain",
    "OGD": "Mountain",

    # Hawaii
    "HNL": "Hawaii", "OGG": "Hawaii", "KOA": "Hawaii",
    "ITO": "Hawaii", "LIH": "Hawaii", "LNY": "Hawaii",
    "MKK": "Hawaii",

    # Alaska
    "ANC": "Alaska", "FAI": "Alaska", "JNU": "Alaska",
    "KTN": "Alaska", "SIT": "Alaska", "BET": "Alaska",
    "BRW": "Alaska", "OME": "Alaska", "OTZ": "Alaska",
    "WRG": "Alaska", "PSG": "Alaska", "CDV": "Alaska",
    "ADQ": "Alaska", "YAK": "Alaska", "DUT": "Alaska",
    "SCC": "Alaska", "AKN": "Alaska", "DLG": "Alaska",
    "GST": "Alaska", "ADK": "Alaska",

    # Caribbean
    "SJU": "Caribbean", "STT": "Caribbean",
    "STX": "Caribbean", "BQN": "Caribbean",
    "MAZ": "Caribbean",
}

region_colors = {
    "Northeast":  "#1f77b4",
    "Southeast":  "#2ca02c",
    "Midwest":    "#ff7f0e",
    "Southwest":  "#d62728",
    "West":       "#9467bd",
    "Mountain":   "#8c564b",
    "Hawaii":     "#e377c2",
    "Alaska":     "#17becf",
    "Caribbean":  "#bcbd22",
    "Unknown":    "#999999",
}

# Add Caribbean to the color legend
region_colors = {
    "Northeast":  "#1f77b4",   # blue
    "Southeast":  "#2ca02c",   # green
    "Midwest":    "#ff7f0e",   # orange
    "Southwest":  "#d62728",   # red
    "West":       "#9467bd",   # purple
    "Mountain":   "#8c564b",   # brown
    "Hawaii":     "#e377c2",   # pink
    "Alaska":     "#17becf",   # cyan
    "Caribbean":  "#bcbd22",   # yellow-green
    "Unknown":    "#999999",   # grey
}
region_colors = {
    "Northeast": "#1f77b4",   # blue
    "Southeast": "#2ca02c",   # green
    "Midwest":   "#ff7f0e",   # orange
    "Southwest": "#d62728",   # red
    "West":      "#9467bd",   # purple
    "Mountain":  "#8c564b",   # brown
    "Hawaii":    "#e377c2",   # pink
    "Alaska":    "#17becf",   # cyan
    "Unknown":   "#999999",   # grey
}

# 4. Compute strength (weighted degree)
strength_dict = dict(G.degree(weight="weight"))
nx.set_node_attributes(G, strength_dict, "strength")

top10     = sorted(strength_dict, key=strength_dict.get, reverse=True)[:10]
top10_set = set(top10)

# 5. Set labels, colors, sizes
for n in G.nodes:
    region   = airport_region.get(n, "Unknown")
    strength = G.nodes[n].get("strength", 0)

    G.nodes[n]["label"] = n if n in top10_set else ""
    G.nodes[n]["group"] = region
    G.nodes[n]["color"] = region_colors.get(region, "#999999")
    G.nodes[n]["size"]  = max(2, np.log1p(strength) * 3)  

# 6. Clean node / edge attrs
for n, attrs in G.nodes(data=True):
    for k, v in list(attrs.items()):
        try:
            if pd.isna(v):
                G.nodes[n][k] = None
        except (TypeError, ValueError):
            pass

for u, v, attrs in G.edges(data=True):
    for k, val in list(attrs.items()):
        try:
            if pd.isna(val):
                G.edges[u, v][k] = 0
        except (TypeError, ValueError):
            pass

# 7. Convert for netwulf
graph_data = nx.json_graph.node_link_data(G)
if "edges" in graph_data:
    graph_data["links"] = graph_data.pop("edges")

# 8. Top-10 airports table
top10_table = pd.DataFrame({
    "IATA":            top10,
    "region":          [airport_region.get(n, "Unknown") for n in top10],
    "strength":        [G.nodes[n].get("strength") for n in top10],
    "avg_airtime":     [G.nodes[n].get("avg_air_time") for n in top10],
    "avg_distance":    [G.nodes[n].get("avg_distance") for n in top10],
    "avg_dep_delay":   [G.nodes[n].get("avg_dep_delay") for n in top10],
    "avg_arr_delay":   [G.nodes[n].get("avg_arr_delay") for n in top10],
    "cancelled_origin_count":   [G.nodes[n].get("cancelled_origin_count") for n in top10],
    "diverted_origin_count":   [G.nodes[n].get("diverted_origin_count") for n in top10],
})
print(top10_table.to_string(index=False))

# 9. Region color legend
print("\nRegion color legend:")
for region, color in region_colors.items():
    print(f"  {region}: {color}")

# 10. Visualize
config = {
    "preset": "Default",
    "closed": False,
    "remembered": {
        "Default": {
            "0": {
                "zoom": 1,
                "node_charge": -7.2,
                "node_gravity": 0.46,
                "link_distance": 14.4,
                "link_distance_variation": 0,
                "node_collision": True,
                "wiggle_nodes": False,
                "freeze_nodes": False,
                "node_fill_color": "#79aaa0",
                "node_stroke_color": "#555555",
                "node_label_color": "#000000",
                "node_size": 17,
                "node_stroke_width": 1,
                "node_size_variation": 0.5,
                "display_node_labels": True,
                "scale_node_size_by_strength": True,
                "link_color": "#7c7c7c",
                "link_width": 2,
                "link_alpha": 0.4,
                "link_width_variation": 0.5,
                "display_singleton_nodes": False,
                "min_link_weight_percentile": 0,
                "max_link_weight_percentile": 1,
            }
        }
    },
    "folders": {
        "Input/output":  {"preset": "Default", "closed": False, "folders": {}},
        "Physics":       {"preset": "Default", "closed": False, "folders": {}},
        "Nodes":         {"preset": "Default", "closed": False, "folders": {}},
        "Links":         {"preset": "Default", "closed": False, "folders": {}},
        "Thresholding":  {"preset": "Default", "closed": False, "folders": {}},
    },
}

stylized_network, config = visualize(graph_data, config=config)

IATA    region  strength  avg_airtime  avg_distance  avg_dep_delay  avg_arr_delay  cancelled_origin_count  diverted_origin_count
 ORD   Midwest   1388214       105.26       1216.27           7.84           5.53                 15285.0                 1104.0
 DFW Southwest   1228058       108.28       1235.17           5.61           1.01                  8413.0                  884.0
 ATL Southeast   1201228        84.20       1029.34           7.90           5.82                  6794.0                  950.0
 LAX      West    812923       131.84       1617.66           4.29           0.75                  4480.0                  510.0
 PHX Southwest    687674       118.53       1453.29           6.93           1.57                  3444.0                  569.0
 IAH Southwest    609715       113.04       1347.36           3.25           2.37                  1801.0                  353.0
 MSP   Midwest    572741       112.53       1310.30           4.31           0.91                

In [118]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
import numpy as np

fig, ax = plt.subplots(figsize=(18, 13))
ax.set_facecolor("white")
ax.set_axis_off()

nodes = stylized_network["nodes"]
links = stylized_network["links"]

# 1. Position lookup
pos = {node["id"]: (node["x"], node["y"]) for node in nodes}

def resolve_id(ref):
    return ref if isinstance(ref, str) else nodes[ref]["id"]

# 2. Build edge segments + weights
segments     = []
edge_weights = []

for link in links:
    src = resolve_id(link["source"])
    tgt = resolve_id(link["target"])
    if src in pos and tgt in pos:
        segments.append([pos[src], pos[tgt]])
        edge_weights.append(link.get("weight", 1))

edge_weights = np.array(edge_weights, dtype=float)

# 3. Draw edges colored by flight count
norm = mcolors.LogNorm(vmin=edge_weights.min(), vmax=edge_weights.max())
cmap = cm.plasma

lc = LineCollection(
    segments,
    cmap       = cmap,
    norm       = norm,
    linewidths = 0.6,
    alpha      = 0.5,
    zorder     = 1,
)
lc.set_array(edge_weights)
ax.add_collection(lc)

def strength_to_size(s, scale=0.01, power=3):
    """
    Log compresses the range, then raises to a power to
    stretch the gap between small and large nodes.
    - Increase `power` for more contrast (try 1.8–2.5)
    - Increase `scale` to make all nodes uniformly bigger
    """
    log_s = max(2.0, np.log1p(float(s)) * 3.0)
    return (log_s ** power) * scale

for node in nodes:
    nid      = node["id"]
    x, y     = node["x"], node["y"]
    color    = node.get("color", "#999999") 
    strength = G.nodes[nid].get("strength", 1) if nid in G.nodes else 1

    ax.scatter(
        x, y,
        color      = color,  
        s          = strength_to_size(strength),
        zorder     = 3,
        edgecolors = "#333333",
        linewidths = 0.3,
    )

# 5. Label top-10 hubs
for node in nodes:
    if node["id"] in top10_set:
        ax.annotate(
            node["id"],
            xy         = (node["x"], node["y"]),
            fontsize   = 14,
            fontweight = "bold",
            ha         = "center",
            va         = "bottom",
            color      = "#000000",
            zorder     = 5,
        )

# 6. Colorbar
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar_ax = fig.add_axes([0.15, 0.04, 0.50, 0.022])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Flight count (log scale)", fontsize=14, labelpad=6)
cbar.ax.tick_params(labelsize=8)

# 7. Region legend
present_regions = {airport_region.get(n, "Unknown") for n in G.nodes()}

region_legend_elements = [
    Line2D([0], [0], marker="o", color="w",
           label=region, markerfacecolor=color, markersize=9)
    for region, color in region_colors.items()
    if region in present_regions
]

region_legend = ax.legend(
    handles        = region_legend_elements,
    title          = "US Region",
    loc            = "lower right",
    bbox_to_anchor = (1.0, 0.0),
    frameon        = True,
    framealpha     = 0.9,
    fontsize       = 10,
    title_fontsize = 10,
)
ax.add_artist(region_legend)


# 9. Save
ax.autoscale_view()
ax.set_title("Airport Graph - 2002_2003", fontsize=16, fontweight="bold", pad=15)
fig.subplots_adjust(bottom=0.12)
fig.savefig("airport_network_edge_strength.png", dpi=300, bbox_inches="tight")
plt.show()

/var/folders/qb/2g2dn0_s7w9dxjgx4qyk86dh0000gn/T/ipykernel_19738/665470894.py:125: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
